# Optymalizacja

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import sympy as sp
import folium

#### Funkcje do wizualizacji - możesz zwinąć po wykonaniu komórki z kodem

In [ ]:
def generate_noisy_poly_samples(degree=20,
                                n_samples=300,
                                x_range=(-1.0, 1.0),
                                low=-1.0,
                                high=1.0,
                                noise_std=0.1,
                                seed=42):
    """
    Generate noisy samples from a random polynomial of given degree.

    Returns:
      poly_coeffs : ndarray shape (degree+1,)  -- coefficients in numpy.polyval order (highest -> constant)
      x_samples   : ndarray shape (n_samples,)
      y_noisy     : ndarray shape (n_samples,)
      y_true      : ndarray shape (n_samples,)
      df_samples  : pandas.DataFrame with columns ['x', 'y']
    """
    rng = np.random.default_rng(seed)

    # random coefficients: highest-degree first (as required by np.polyval)
    poly_coeffs = rng.uniform(low=low, high=high, size=(degree + 1,))
    # ensure leading coefficient is not (near) zero
    if np.abs(poly_coeffs[0]) < 1e-12:
        poly_coeffs[0] += np.sign(rng.normal()) * 1e-3
    poly_coeffs = poly_coeffs + np.linspace(0, degree/2, degree+1)[::-1]
    # sample x uniformly and sort for nicer plotting
    x_samples = rng.uniform(x_range[0], x_range[1], size=n_samples)
    x_samples = np.sort(x_samples)

    # evaluate polynomial and add Gaussian noise
    y_true = np.polyval(poly_coeffs, x_samples)
    y_noisy = y_true + rng.normal(loc=0.0, scale=noise_std, size=n_samples)

    # assemble dataframe (doesn't overwrite existing `data`)
    poly20_df = pd.DataFrame({'x': x_samples, 'y': y_noisy})

    return poly_coeffs, x_samples, y_noisy, y_true, poly20_df


def make_parabolic_example(f, df, learning_rate, compute_new_x, start_x=0.0, range_x=(-1, 7), iterations=25):
    # Najpierw obliczamy całą historię kroków, a dopiero potem ją animujemy.
    history_x = [start_x]
    current_x = start_x
    for _ in range(iterations):
        current_x = compute_new_x(current_x, df, learning_rate)
        # gradient = df(current_x)
        # current_x = current_x - learning_rate * gradient
        history_x.append(current_x)
    fig, ax = plt.subplots(figsize=(10, 6))
    x_vals = np.linspace(range_x[0], range_x[1], 200)
    y_vals = f(x_vals)

    ax.plot(x_vals, y_vals, 'b-', linewidth=2, alpha=0.6, label='Funkcja kosztu')
    ax.set_xlim(range_x)
    ax.set_ylim(0, 25)
    ax.set_xlabel('Parametr x')
    ax.set_ylabel('Koszt f(x)')
    ax.set_title('Animacja Gradient Descent')
    ax.grid(True)

    point, = ax.plot([], [], 'ro', markersize=12, label='Aktualna pozycja', zorder=5)
    tangent_line, = ax.plot([], [], 'g--', linewidth=2, label='Styczna (Gradient)')
    text_info = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=12,
                        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))


    def update(frame):
        x_curr = history_x[frame]
        y_curr = f(x_curr)
        grad = df(x_curr)
        point.set_data([x_curr], [y_curr])
        # Równanie stycznej: y - y0 = m(x - x0)  =>  y = m(x - x0) + y0
        tangent_range = 1.5 # Długość linii w lewo i w prawo
        x_tan = np.linspace(x_curr - tangent_range, x_curr + tangent_range, 10)
        y_tan = grad * (x_tan - x_curr) + y_curr
        tangent_line.set_data(x_tan, y_tan)

        # 3. Zaktualizuj tekst
        text_info.set_text(f'Iteracja: {frame}\n'
                        f'x = {x_curr:.4f}\n'
                        f'Gradient = {grad:.4f}')

        return point, tangent_line, text_info

    anim = FuncAnimation(fig, update, frames=len(history_x), interval=500, blit=True)
    plt.close()
    print("Generowanie animacji...")
    return HTML(anim.to_jshtml())


def make_higher_order_example(f_num, f_grad_num, scenarios, learning_rate, compute_new_x, steps=50, show_trace=True):
    all_histories = []

    for s in scenarios:
        history = [s["start"]]
        curr_x = s["start"]
        for _ in range(steps):
            curr_x = compute_new_x(curr_x, f_grad_num, learning_rate)
            history.append(curr_x)
        all_histories.append(history)

    # Przygotowanie wykresu
    fig, ax = plt.subplots(figsize=(10, 6))
    x_vals = np.linspace(-2.3, 2.3, 400)
    y_vals = f_num(x_vals)

    # Tło
    ax.plot(x_vals, y_vals, 'k-', alpha=0.3, linewidth=2, label='f(x)')
    ax.set_title(f"Gradient Descent (Trace: {'ON' if show_trace else 'OFF'})")
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.set_ylim(-1, 9)
    ax.grid(True, alpha=0.3)

    # Inicjalizacja obiektów graficznych
    points_plots = []
    labels_plots = []
    trails_plots = []  # Nowa lista na ślady

    for s in scenarios:
        t, = ax.plot([], [], '-', color=s["color"], linewidth=3, alpha=0.6)
        trails_plots.append(t)
        p, = ax.plot([], [], 'o', color=s["color"], markersize=10)
        points_plots.append(p)
        l = ax.text(0, 0, s["name"], color=s["color"], fontweight='bold', ha='center', fontsize=9)
        labels_plots.append(l)

    def update(frame):
        artists = []
        for i, history in enumerate(all_histories):
            idx = min(frame, len(history)-1)
            x_curr = history[idx]
            y_curr = f_num(x_curr)
            if show_trace:
                path_x = history[:idx+1]
                path_y = f_num(np.array(path_x))
                trails_plots[i].set_data(path_x, path_y)
                artists.append(trails_plots[i])
            points_plots[i].set_data([x_curr], [y_curr])
            artists.append(points_plots[i])
            labels_plots[i].set_position((x_curr, y_curr + 0.6))
            artists.append(labels_plots[i])

        return artists

    anim = FuncAnimation(fig, update, frames=steps, interval=100, blit=True)
    plt.close()

    return HTML(anim.to_jshtml())


def make_2d_parabolic_example(f_num, grads_num, scenarios, learning_rate, compute_new_position_with_gradient, steps=50, show_trace=True):
    all_histories = []
    for s in scenarios:
        pos = s["start"]
        history = [pos] # Lista pozycji [x, y]
        for _ in range(steps):
            pos = compute_new_position_with_gradient(pos, grads_num, learning_rate)
            history.append(pos)
        all_histories.append(np.array(history))

    fig = plt.figure(figsize=(16, 7))

    # --- LEWY PANEL: mapa konturowa ---
    ax2d = fig.add_subplot(1, 2, 1)
    x_range = np.linspace(-2.5, 2.5, 150)
    y_range = np.linspace(-2.5, 2.5, 150)
    X, Y = np.meshgrid(x_range, y_range)
    Z = f_num((X, Y))

    # Rysujemy mapę konturową
    contour = ax2d.contourf(X, Y, Z, levels=35, cmap='viridis')
    fig.colorbar(contour, ax=ax2d, shrink=0.6, label='Funkcja Kosztu')
    ax2d.set_title("Widok z góry (2D)")
    ax2d.set_xlabel("Parametr x")
    ax2d.set_ylabel("Parametr y")

    # --- PRAWY PANEL: wykres 3D ---
    ax3d = fig.add_subplot(1, 2, 2, projection='3d')

    # Rysujemy powierzchnię (tylko raz, jako tło)
    surf = ax3d.plot_surface(X, Y, Z, cmap='viridis', alpha=0.5,
                            edgecolor='none', rstride=2, cstride=2)
    ax3d.set_title("Widok przestrzenny (3D)")
    ax3d.set_xlabel("x")
    ax3d.set_ylabel("y")
    ax3d.set_zlabel("Koszt")
    ax3d.set_zlim(-1.5, 46)
    # Ustawiamy widok, żeby widzieć doliny
    ax3d.view_init(elev=0, azim=-60)

    # Musimy przechowywać obiekty dla 2D i 3D osobno
    lines_2d, points_2d = [], []
    lines_3d, points_3d = [], []

    for s in scenarios:
        l2d, = ax2d.plot([], [], '-', color=s["color"], lw=1.5, alpha=0.8)
        p2d, = ax2d.plot([], [], 'o', color=s["color"], markeredgecolor='black', markersize=8)
        lines_2d.append(l2d)
        points_2d.append(p2d)

        l3d, = ax3d.plot([], [], [], '-', color=s["color"], lw=2, alpha=0.9)
        p3d, = ax3d.plot([], [], [], 'o', color=s["color"], markeredgecolor='black', markersize=8)
        lines_3d.append(l3d)
        points_3d.append(p3d)

    # 6. Pętla Animacji
    def update(frame):
        artists = []

        for i, history in enumerate(all_histories):
            idx = min(frame, len(history)-1)

            # Dane do aktualnej klatki
            # shape (N, 2) -> path_x, path_y
            path_x = history[:idx+1, 0]
            path_y = history[:idx+1, 1]
            curr_x = path_x[-1]
            curr_y = path_y[-1]

            # Obliczamy Z (wysokość) dla widoku 3D
            path_z = f_num((path_x, path_y))
            curr_z = path_z[-1]

            # --- AKTUALIZACJA 2D ---
            if show_trace:
                lines_2d[i].set_data(path_x, path_y)
                artists.append(lines_2d[i])

            points_2d[i].set_data([curr_x], [curr_y])
            artists.append(points_2d[i])

            # --- AKTUALIZACJA 3D ---
            if show_trace:
                lines_3d[i].set_data(path_x, path_y)
                lines_3d[i].set_3d_properties(path_z)
                artists.append(lines_3d[i])

            points_3d[i].set_data([curr_x], [curr_y])
            points_3d[i].set_3d_properties([curr_z])
            artists.append(points_3d[i])

        return artists

    # Generowanie
    print("Generowanie animacji 2D + 3D...")
    anim = FuncAnimation(fig, update, frames=steps, interval=80, blit=True)
    plt.close()

    return HTML(anim.to_jshtml())


def make_3D_higher_order_example(f_num, grads, scenarios, learning_rate, compute_new_position_with_gradient, steps=50, show_trace=True):
    # Pre-kalkulacja ścieżek (Symulacja)
    # Obliczamy wszystko przed animacją dla płynności
    all_histories = []
    for s in scenarios:
        pos = s["start"]
        history = [pos] # Lista pozycji [x, y]
        for _ in range(steps):
            pos = compute_new_position_with_gradient(pos, grads, learning_rate)
            history.append(pos)
        all_histories.append(np.array(history))

    # Przygotowanie Sceny (Figury i Subploty)
    fig = plt.figure(figsize=(16, 7))

    # --- LEWY PANEL: 2D CONTOUR ---
    ax2d = fig.add_subplot(1, 2, 1)
    x_range = np.linspace(-4, 4, 150)
    y_range = np.linspace(-2.5, 2.5, 150)
    X, Y = np.meshgrid(x_range, y_range)
    Z = f_num((X, Y))

    # Rysujemy mapę konturową
    contour = ax2d.contourf(X, Y, Z, levels=35, cmap='viridis')
    fig.colorbar(contour, ax=ax2d, shrink=0.6, label='Funkcja Kosztu')
    ax2d.set_title("Widok z góry (2D)")
    ax2d.set_xlabel("Parametr x")
    ax2d.set_ylabel("Parametr y")

    # --- PRAWY PANEL: 3D SURFACE ---
    ax3d = fig.add_subplot(1, 2, 2, projection='3d')

    # Rysujemy powierzchnię (tylko raz, jako tło)
    # rstride/cstride = 2 dla optymalizacji wydajności renderowania
    surf = ax3d.plot_surface(X, Y, Z, cmap='viridis', alpha=0.5,
                            edgecolor='none', rstride=2, cstride=2)
    ax3d.set_title("Widok przestrzenny (3D)")
    ax3d.set_xlabel("x")
    ax3d.set_ylabel("y")
    ax3d.set_zlabel("Koszt")
    ax3d.set_zlim(-1.5, 6)
    # Ustawiamy widok, żeby widzieć doliny
    ax3d.view_init(elev=0, azim=-60)

    # Inicjalizacja Obiektów Animowanych (Kropki i Linie)
    # Musimy przechowywać obiekty dla 2D i 3D osobno
    lines_2d, points_2d = [], []
    lines_3d, points_3d = [], []

    for s in scenarios:
        # --- Obiekty 2D ---
        l2d, = ax2d.plot([], [], '-', color=s["color"], lw=1.5, alpha=0.8)
        p2d, = ax2d.plot([], [], 'o', color=s["color"], markeredgecolor='black', markersize=8)
        lines_2d.append(l2d)
        points_2d.append(p2d)

        # --- Obiekty 3D ---
        # W 3D używamy ax.plot zamiast ax.scatter dla łatwiejszej animacji
        l3d, = ax3d.plot([], [], [], '-', color=s["color"], lw=2, alpha=0.9)
        p3d, = ax3d.plot([], [], [], 'o', color=s["color"], markeredgecolor='black', markersize=8)
        lines_3d.append(l3d)
        points_3d.append(p3d)

    # Pętla Animacji
    def update(frame):
        artists = []

        for i, history in enumerate(all_histories):
            idx = min(frame, len(history)-1)

            # Dane do aktualnej klatki
            # shape (N, 2) -> path_x, path_y
            path_x = history[:idx+1, 0]
            path_y = history[:idx+1, 1]
            curr_x = path_x[-1]
            curr_y = path_y[-1]

            # Obliczamy Z (wysokość) dla widoku 3D
            path_z = f_num((path_x, path_y))
            curr_z = path_z[-1]

            # --- AKTUALIZACJA 2D ---
            if show_trace:
                lines_2d[i].set_data(path_x, path_y)
                artists.append(lines_2d[i])

            points_2d[i].set_data([curr_x], [curr_y])
            artists.append(points_2d[i])

            # --- AKTUALIZACJA 3D ---
            if show_trace:
                lines_3d[i].set_data(path_x, path_y)
                lines_3d[i].set_3d_properties(path_z)
                artists.append(lines_3d[i])

            points_3d[i].set_data([curr_x], [curr_y])
            points_3d[i].set_3d_properties([curr_z])
            artists.append(points_3d[i])

        return artists

    # Generowanie (może chwilę potrwać)
    print("Generowanie animacji 2D + 3D...")
    anim = FuncAnimation(fig, update, frames=steps, interval=80, blit=True)
    plt.close()

    return HTML(anim.to_jshtml())




def make_linear_3d_example(predict_func, grad_funcs, weights, loss_func, learning_rate, iterations=300):
    # Generowanie danych
    X_data = np.linspace(-2, 2, 50)
    noise = np.random.normal(0, 1.0, X_data.shape)
    # True function: y = 3x + 2
    Y_data = 3 * X_data + 2 + noise

    history_weights = []
    mses = []
    for i in range(iterations):
        # Zapisz obecny stan wag (kopiujemy słownik)
        history_weights.append(weights.copy())

        # Obliczamy gradienty dla CAŁEGO zbioru danych (średnia)
        current_grads = {k: 0.0 for k in weights}

        # Uwaga: W prawdziwym ML robi się to wektorowo, tu pętla dla jasności
        # Sumujemy gradienty ze wszystkich punktów danych

        # Wartości argumentów do funkcji gradientu
        args = (X_data, Y_data, weights['a'], weights['b'])

        mses.append(np.mean(loss_func(args)))
        for key in weights:
            # Obliczamy gradient dla wszystkich punktów naraz (numpy)
            g_values = grad_funcs[key](args)
            # Uśredniamy gradient (Mean Squared Error)
            current_grads[key] = np.mean(g_values)

        # Aktualizacja wag (Gradient Descent Step)
        for key in weights:
            weights[key] = weights[key] - learning_rate *current_grads[key]

    # --- Tworzymy wizualizację funkcji kosztu ---
    # We create a grid of 'a' and 'b' values to visualize the "bowl"
    a_range = np.linspace(-6, 6, 40)
    b_range = np.linspace(-2, 8, 40)
    A_grid, B_grid = np.meshgrid(a_range, b_range)
    MSE_grid = np.zeros_like(A_grid)

    # Calculate MSE for every point on the grid
    for i in range(A_grid.shape[0]):
        for j in range(A_grid.shape[1]):
            a_val, b_val = A_grid[i, j], B_grid[i, j]
            y_p = predict_func((X_data, a_val, b_val))
            MSE_grid[i, j] = np.mean((Y_data - y_p)**2)

    # --- 4. VISUALIZATION ---
    fig = plt.figure(figsize=(14, 6))

    # -- Subplot 1: 2D Line Fitting --
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.scatter(X_data, Y_data, color='blue', alpha=0.5, label='Data Points')
    line, = ax1.plot([], [], 'r-', linewidth=3, label='Model (y=ax+b)')
    ax1.set_xlim(-2.5, 2.5)
    ax1.set_ylim(-5, 10)
    ax1.set_title("2D: Fitting the Line")
    ax1.set_xlabel("x")
    ax1.set_ylabel("y")
    ax1.legend()

    text_info = ax1.text(0.05, 0.90, '', transform=ax1.transAxes, bbox=dict(facecolor='white', alpha=0.8))

    # -- Subplot 2: 3D Loss Landscape --
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    # Plot the wireframe surface (The "Valley")
    ax2.plot_surface(A_grid, B_grid, MSE_grid, cmap='viridis', alpha=0.4, edgecolor='none')
    ax2.set_title("3D: Gradient Descent Path")
    ax2.set_xlabel("Parameter a (Slope)")
    ax2.set_ylabel("Parameter b (Intercept)")
    ax2.set_zlabel("Loss (MSE)")

    # The point representing our current model in 3D
    scatter_3d = ax2.scatter([], [], [], color='red', s=50, label='Current State')

    # Plot the trajectory (the path taken so far)
    trajectory_line, = ax2.plot([], [], [], color='black', linestyle='--', linewidth=1)

    def update(frame):
        w = history_weights[frame]
        current_mse = mses[frame]

        # 1. Update 2D Line
        x_plot = np.linspace(-2.5, 2.5, 100)
        y_plot = w['a'] * x_plot + w['b']
        line.set_data(x_plot, y_plot)

        text_info.set_text(f"Iter: {frame}\n"
                           f"a (slope): {w['a']:.2f}\n"
                           f"b (intercept): {w['b']:.2f}\n"
                           f"MSE: {current_mse:.2f}")

        # 2. Update 3D Point & Trajectory
        # Get history up to this frame for the trail
        past_a = [history_weights[k]['a'] for k in range(frame+1)]
        past_b = [history_weights[k]['b'] for k in range(frame+1)]
        past_mse = mses[:frame+1]

        # Update the red dot (current position)
        scatter_3d._offsets3d = ([w['a']], [w['b']], [current_mse])

        # Update the black dashed line (path)
        trajectory_line.set_data(past_a, past_b)
        trajectory_line.set_3d_properties(past_mse)

        return line, text_info, scatter_3d, trajectory_line

    print("Generating 3D animation...")
    anim = FuncAnimation(fig, update, frames=len(history_weights), interval=100, blit=False)
    plt.close()

    return HTML(anim.to_jshtml())


def make_polynomial_example(predict_func, grad_funcs, weights, loss_func, learning_rate, iterations=300):
    # Ograniczamy zakres x, bo duże potęgi x rosną bardzo szybko.
    # Może to doprowadzić do bardzo dużych wartości gradientu i niestabilności numerycznej
    # Tzw. eksplodującego gradientu
    X_data = np.linspace(-1.6, 1.6, 50)
    # Dodajemy szum (np. błędy pomiarowe), żeby zadanie było realistyczne
    noise = np.random.normal(0, 0.5, X_data.shape)
    # Prawdziwa funkcja: x^6 - 5x^4 + 7x^2
    Y_data = X_data**6 - 5*X_data**4 + 7*X_data**2 + noise

    history_weights = [] # Tu będziemy zapisywać postępy do animacji
    mses = []
    for i in range(iterations):
        # Zapisz obecny stan wag (kopiujemy słownik)
        history_weights.append(weights.copy())

        # Obliczamy gradienty dla CAŁEGO zbioru danych (średnia)
        current_grads = {k: 0.0 for k in weights}

        # Uwaga: W prawdziwym ML robi się to wektorowo, tu pętla dla jasności
        # Sumujemy gradienty ze wszystkich punktów danych

        # Wartości argumentów do funkcji gradientu
        args = (X_data, Y_data, weights['a'], weights['b'], weights['c'],
                weights['d'], weights['e'], weights['f'], weights['g'])

        mses.append(np.mean(loss_func(args)))
        for key in weights:
            # Obliczamy gradient dla wszystkich punktów naraz (numpy)
            g_values = grad_funcs[key](args)
            # Uśredniamy gradient (Mean Squared Error)
            current_grads[key] = np.mean(g_values)

        # Aktualizacja wag (Gradient Descent Step)
        for key in weights:
            weights[key] = weights[key] - learning_rate *current_grads[key]

    # --- 4. WIZUALIZACJA (Animacja) ---
    fig, ax = plt.subplots(figsize=(10, 6))

    # Rysujemy prawdziwe dane (niebieskie kropki)
    ax.scatter(X_data, Y_data, color='blue', alpha=0.5, label='Pomiary (Dane)')
    ax.set_ylim(-2, 8)
    ax.set_xlim(-1.8, 1.8)
    ax.set_title("Dopasowywanie wielomianu 6. stopnia")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    # Linia naszego modelu (zacznie się losowo, skończy dopasowana)
    line, = ax.plot([], [], 'r-', linewidth=3, label='Model AI')
    epoch_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.8))
    mse_text = ax.text(0.02, 0.90, '', transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.8))
    equation_text = ax.text(0.02, 0.02, '', transform=ax.transAxes, fontsize=8, verticalalignment='bottom')

    ax.legend(loc='upper right')

    def update(frame):
        # Pobieramy wagi z danej epoki
        w = history_weights[frame]
        mse = mses[frame]
        # Generujemy linię predykcji dla tych wag
        x_plot = np.linspace(-1.8, 1.8, 200)
        # args: x, a, b, c, d, e, f, g
        y_plot = predict_func((x_plot, w['a'], w['b'], w['c'], w['d'], w['e'], w['f'], w['g']))

        line.set_data(x_plot, y_plot)
        epoch_text.set_text(f'Epoka: {frame}/{iterations}')
        mse_text.set_text(f"MSE: {mse:.3e}")

        # Wyświetlanie aktualnego równania (formatowanie)
        eq_str = (f"h(x) = {w['a']:.2f}x^6 + {w['b']:.2f}x^5 + {w['c']:.2f}x^4 + "
                    f"{w['d']:.2f}x^3 + {w['e']:.2f}x^2 + {w['f']:.2f}x + {w['g']:.2f}")
        equation_text.set_text(eq_str)

        return line, epoch_text, equation_text

    print("Generowanie animacji dopasowywania...")
    anim = FuncAnimation(fig, update, frames=len(history_weights), interval=50, blit=True)
    plt.close()

    return HTML(anim.to_jshtml())



## Czym jest Optymalizacja?

Codzienne zycie nie jest łatwe. Wiąze się z wieloma zadaniami, które trzeba zaplanować i  wykonać, np.
* Jak dojechać z domu na uczelnię w najkrótszym czasie?
* Jak spakować się w sam plecak na weekendowy wyjazd, zeby nie musieć brać ze sobą walizki?
* Jak poradzić sobie, gdy pieniędzy na dany miesiąc (wypłaty, stypendium, kieszonkowego) zostało juz mało, a do kolejnej wypłaty został jeszcze tydzień?
* Jak zaplanować gotowanie, aby ziemniaki i kotlet były gotowe w tym samym momencie?

Prawdopodobnie choć jeden z tych problemów jest Ci znany. Rozwiązując je wykonujemy *optymalizację*, czyli przeprowadzamy **proces znajdowania najlepszego rozwiązania**.
Optymalizacja nie jest tylko domeną sztucznej inteligencji - to naturalny proces, który wykonujemy codziennie, dążąc do osiągnięcia najlepszego rezultatu przy ograniczonych zasobach (czasie, pieniądzach, energii).

* Planujesz trasę na uczelnię lub wykorzystujesz w tym celu dedykowane narzędzia.
    * **Cel:** Dojechać z domu na uczelnię.
    * **Przedmiot optymalizacji:** Czas podróży.
    * **Zmienne:** Przebieg trasy.
    * **Ograniczenia:** Korki, roboty drogowe, limity prędkości.


* Wyjeżdżasz na weekend i chcesz wziąć tylko plecak.
    * **Cel:** Zapakować jak najwięcej potrzebnych przedmiotów.
    * **Przedmiot optymalizacji:** Użyteczność zabranych rzeczy lub objętość upchniętych ubrań.
    * **Zmienne:** Przedmioty które zabierasz i ich ułożenie w plecaku (np. zwinięcie w rulon, przegroda w którą je spakujemy).
    * **Ograniczenia:** Wymiary plecaka, wygoda noszenia go.


* Musisz przezyć jeszcze tydzień, a pieniędzy masz juz niewiele.
    * **Cel:** Przezyć, nie być głodnym.
    * **Co optymalizujemy (szukamy kompromisu):** Koszt zakupionych przedmiotów, dostarczana przez nie energia i wartości odzywcze, stosunek jakości do ceny.
    * **Ograniczenia:** Suma posiadanych w portfelu pieniędzy.


* Chcesz podać niedzielny obiad - rosół, kotleta, ziemniaki i dodatek (np. surówkę) tak, aby pierwsza była zupa, a później drugie danie. Żeby było smacznie, jedzenie powinno być ciepłe.
    * **Cel:** Zjeść ciepły obiad dwudaniowy.
    * **Co optymalizujemy (minimalizujemy):** Czas przygotowania, róznicę czasu gotowości dań, różnicę temperatur między daniami.
    * **Zmienne:** Moment rozpoczęcia gotowania składników (rosół potrzebuje 4 godziny, ziemniaki potrzebują 20 min, kotlet 5 min, a surówkę juz masz).
    * **Ograniczenia:** Liczba palników na kuchence, dostępne narzędzia, czas przygotowania poszczególnych składników, czas potrzebny na zjedzenie rosołu.

Optymalizacja nie musi polegać na znalezieniu najlepszego wyniku - czasem rozwiązanie znalezione szybko moze okazać się akceptowalnie dobre (np. zrezygnowanie z zapakowania jednej koszulki, lub zostawienie ziemniaków po ugotowaniu w garnku, aby nie ostygły zbyt szybko). W takim przypadku dodatkowo optymalizujemy koszt znalezienia rozwiązania. W podobny sposób działa mózg. Jest on tzw. "skąpcem poznawczym"<sup>1</sup> i woli uprościć problem lub zdać się na szybkie (wydajne) rozwiązanie, niż sporym wysiłkiem szukać rozwiązania idealnego.

Daniel Kahneman, autor ksiązki "Pułapki myślenia. O myśleniu szybkim i wolnym" (Thinking, Fast and Slow<sup>2</sup>) podzielił nasze myślenie na dwa systemy.

* System 1 (Szybki, Instynktowny, Tani) działa automatycznie, bez znaczącego wysiłku umysłowego. To np. wynik działania 2+2, obieranie ziemniaków, jazda samochodem pustą drogą. Są to działania relatywnie proste, wykonane wiele razy, które mózg dobrze zna. Mózg nie "liczy" wyniku 2+2, on go "pamięta". W informatyce do takich zadań wykorzystujemy tzw. Lookup Table (tablica wyników) lub Cache. To jest optymalizacja przez zapamiętywanie - memoizacja.

* System 2 (Wolny, Analityczny, Drogi)w ymaga uwagi i skupienia. Zuzywa przy tym duzo energii. To np. wynik działania $2^\pi$, parkowanie równoległe "na styk", nauka nowej gry planszowej. Mózg musi wykonać wszystkie procesy krok po kroku. W informatyce byłby to skomplikowany algorytm, np. A* bez wykorzystania Cache.

Mózg optymalizuje swoje zasoby, starając się jak najczęściej używać Systemu 1, pozostawiając System 2 tylko na nadzwyczajne (nieznane / niewystarczająco znane) sytuacje.

Podobnie mozna optymalizować uzycie zasobów w komputerze (czas procesora, pamięć RAM, energia). Przykładowo wiele algorytmów (np. rózne algorytmy sortowania) było ulepszanych tak, by miały niższą złożoność obliczeniową, czyli lepiej się skalowały wraz z rozmiarem problemu. Nowoczesne kompilatory są pisane tak, aby optymalizować kod programisty (np. usuwają martwy kod, rozwijają pętle), by program działał szybciej. Często wyniki obliczeń w komputerze są zapamiętywane (np. Cache), aby móc szybko sięgnąć po nie gdy są potrzebne, zamiast za kazdym razem je obliczać lub szukać.

## Optymalizacja w Sztucznej Inteligencji

Choć w Sztucznej Inteligencji (SI) pojęcie optymalizacji takze jest niezwykle szerokie, w ramach tej listy zajmiemy się optymalizacją parametrów funkcji tak, aby jak najlepiej odwzorowywała ona dostarczone dane. Co takie działanie ma na celu?

Każdy model SI, niewazne czy słuzy do rozpoznawania obrazów, tłumaczenia języka, czy przewidywania cen akcji, w rzeczywistości jest zbiorem parametrów w oparciu o które wykonywane są obliczenia. Aby model był skuteczny, powinien dobrze oddawać rzeczywistość. To z kolei wymaga, aby funkcja którą jest dany była jak najlepiej dopasowana do tej rzeczywistości. Skąd jednak wiemy, jakie parametry dobrać, aby uzyskać "najlepsze" dopasowanie?

Tym problemem zajmuje się optymalizacja.

Dopasowaniem funkcji do danych zajmowaliśmy się juz w ramach pierwszego laboratorium. Próbowaliśmy znaleźć takie parametry modelu (wtedy wielomianu czwartego stopnia), aby jak najlepiej oddawał on zachowanie danych w określonym przedziale. Przypomnijmy szybko ten problem.

In [ ]:
data = pd.read_csv(f"data/example_1.csv")
x = data["x"]
y = data["y"]

poly_coeffs = {
    "a_4": -0.07,
    "a_3": 0.05,
    "a_2": -0.89,
    "a_1": -0.02,
    "a_0": 1.00
}

poly_coeffs = [i for _, i in sorted(poly_coeffs.items(), key=lambda x: x[0])]
y_best = sum(c * x**i for i, c in enumerate(poly_coeffs))
mse = np.mean((y - y_best) ** 2)

sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))
sns.scatterplot(x=x, y=y, color="gray", s=20, label="Dane")
sns.lineplot(x=x, y=y_best, color="red", linestyle="--", label="Model")
plt.title(f"Porównanie modeli. MSE: {mse:.2e}")
plt.xlabel("x")
plt.ylabel("y")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

Znalezienie parametrów niezłego modelu było stosunkowo proste:
1. Wykonaliśmy wstępne dopasowanie ręcznie
2. Znaleźliśmy dokładniejsze dopasowanie metodą przeglądu zupełnego w założonym zakresie wartości parametrów

Czy takie podejście zawsze będzie skuteczne?

In [ ]:
data = pd.read_csv(f"data/example_2.csv")
x = data["x"]
y = data["y"]
y_best = sum(c * x**i for i, c in enumerate(poly_coeffs))

sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))
sns.scatterplot(x=x, y=y, color="gray", s=20, label="Dane")
sns.lineplot(x=x, y=y_best, color="red", linestyle="--", label="Model")
plt.title(f"Pełny wykres krzywej naprężenia.")
plt.xlabel("x")
plt.ylabel("y")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

W przypadku kształtu zbliżonego do paraboli znalezienie współczynników było proste. W przypadku kształtu mniej oczywistego, który rysuje nam się z całej próby, współczynniki nie będą tak oczywiste. Potrzebujemy zatem metody automatycznego określania parametrów takiej funkcji. Do takich celów wykorzystujemy optymalizację.

## Optymalizacja z wykorzystaniem pochodnej

W ramach tej listy zajmiemy się jednym z **najpowszechniejszych algorytmów optymalizacyjnych** w całym uczeniu maszynowym: **Gradient Descent** (GD), po polsku nazywany **algorytmem gradientu prostego**.

Angielska nazwa algorytmu ma 2 części nazwy: **Gradient** i **Descent**.

Zajmijmy się najpierw Descent czyli zejściem / spadkiem. W optymalizacji zazwyczaj szukamy minimum funkcji - taka jest konwencja. To tak, jakby szukać doliny będąc w górach - trzeba zejść w dół po zboczu (gdybyśmy maksymalizowali, szukalibyśmy szczytu i szli w górę). Aby zejść w dół, musimy wiedzieć, w którą stronę teren opada. Jakie matematyczne narzędzie służy do sprawdzenia nachylenia funkcji?

Tutaj mamy do czynienia z drugą częścią nazwy, czyli **gradientem funkcji**. Formalnie jest to wektor złożony z pochodnych cząstkowych funkcji wielu zmiennych. Póki co, traktujmy go jako uogólnienie **pochodnej**. Z matematyki ze szkoły średniej wiesz, że pochodna służy do określenia nachylenia funkcji:

* Jeśli jest dodatnia: $f'(x) > 0$, funkcja rośnie wraz ze wzrostem $x$
* Jeśli jest ujemna $f'(x) < 0$, funkcja maleje wraz ze wzrostem $x$
* Jeśli $f'(x) = 0$, to funkcja nie rośnie ani nie maleje. Mamy do czynienia z maksimum (szczytem) lub minimum (doliną) funkcji.

Te własności pochodnej pozwalają nam obrać prostą, intuicyjną strategię:

1.  Znajdujemy się w punkcie $x_i$.
2.  Sprawdzamy nachylenie w danym miejscu (czyli obliczamy pochodną $f'(x_i)$).
3.  Jeśli nachylenie jest **dodatnie** ($f'(x_i) > 0$), to znaczy, że wraz ze wzrostem $x$ rośnie też $y$. "Idąc w prawo", idziemy w górę. Aby dotrzeć do minimum, czyli np. zejść z góry, musimy iść w **lewo**.
4.  Jeśli nachylenie jest **ujemne** ($f'(x_i) < 0$), to znaczy, że wraz ze wzrostem $x$ maleje $y$, czyli idąc w prawo idziemy w dół. Aby dotrzeć do minimum, musimy iść w **prawo**.

Oba te przypadki można opisać jednym wzorem:

$$x_{i+1} = x_i - \eta \cdot f'(x_i)$$

Gdzie:
* $x_{i+1}$ to nasze następne położenie.
* $x_i$ to nasze obecne położenie.
* $f'(x_i)$ to pochodna w obecnym punkcie.
* $\eta$ to **współczynnik uczenia** (ang. *learning rate*). Mówi on, jak **duży krok** robimy.
    * Jeśli $\eta$ jest za małe, to tak, jakby poruszać się malutkimi kroczkami - zejście zajmie wieczność.
    * Jeśli $\eta$ jest za duże, to tak, jakby poruszać się wielkimi skokami - można "przeskoczyć" dolinę i wylądować na górze po drugiej stronie doliny. Jeżeli skok będzie wystarczająco duży, możemy znaleźć się jeszcze wyżej niż byliśmy.

### Zadanie
(1 pkt)

Na podstawie wzoru powyżej, uzupełnij funkcję `compute_new_x` i wykonaj komórkę. Upewnij się, że funkcja działa poprawnie (pomoże w tym wygenerowana animacja).

In [ ]:
def compute_new_x(x, df, learning_rate):
    """
    Wykonuje pojedynczy krok algorytmu Gradient Descent dla funkcji jednej zmiennej.

    Args:
        x (float): Aktualna pozycja (wartość parametru), w której się znajdujemy.
        df (callable): Funkcja obliczająca pochodną (nachylenie) w danym punkcie.
                            Musi przyjmować jeden argument i zwracać liczbę.
        learning_rate (float): Współczynnik uczenia (eta). Określa wielkość kroku.
                            Decyduje o tym, jak daleko przesuniemy się w jednej iteracji.

    Returns:
        float: Nowa, zaktualizowana pozycja x.
    """
    # Oblicz wartość pochodnej w punkcie x
    derivative_value = df(x)

    # Zwróć nową pozycję x
    return x - learning_rate * derivative_value


learning_rate = 0.1
# Funkcja i jej pochodna
f = lambda x: x**2 - 6*x + 10
df = lambda x: 2*x - 6  # Pochodna: 2x - 6
make_parabolic_example(f, df, learning_rate, compute_new_x)

W rzeczywistości przypadki takie jak powyżej, gdzie funkcja jednoznacznie zbiega do jednego minimum, są rzadkością. Dla większości funkcji obserwujemy zjawisko minimów lokalnych, czyli obecności wielu minimów w krajobrazie funkcji. Jest ono problematyczne, ponieważ chcielibyśmy znajdować najmniejszą wartość w całej dziedzinie - tzw. minimum globalne.

### ZADANIE
(2 pkt.)

Używając kodu poniżej zbadaj:
* W jaki sposób learning_rate wpływa na działanie algorytmu Gradient Descent
* W jaki sposób pozycja początkowa (`start`) wpływa na znajdowane rozwiązania

Zapisz wnioski.

In [ ]:
# --- KONFIGURACJA ---
show_trace = True  # <--- ZMIEŃ NA False, ABY UKRYĆ ŚLAD
steps = 50         # Liczba klatek animacji
learning_rate = 0.1

scenarios = [
    {"name": "Lewy",   "start": -1.9, "color": "red"},
    {"name": "Środkowy", "start": -0.6, "color": "green"},
    {"name": "Prawy",  "start":  1.9, "color": "blue"}
]

# Definicja symboliczna funkcji i gradientu
x = sp.symbols('x')
f_sym = x**6 - 5*x**4 + 7*x**2
f_prime_sym = sp.diff(f_sym, x)

# Konwersja na funkcje numeryczne
f_num = sp.lambdify(x, f_sym, 'numpy')
f_df_num = sp.lambdify(x, f_prime_sym, 'numpy')


make_higher_order_example(f_num, f_df_num, scenarios, learning_rate, compute_new_x, steps, show_trace)


W podanym przykładzie większy learning rate powoduje to, że nie tylko środkowy przykład, ale też prawy i lewy schodzą do minimum globalnego, przy mniejszych wartościach zatrzymują się one w swoich minimach lokalnych. Większy learning rate powoduje większe "skoki". Pozycja startowa ma duży wpyływ na znalezione rozwiązanie, bo w tym przykładzie częściej najlepsze rozwiązanie jest znajdywane przy środkowym punkcie startowym, bo w drodze do minimum globalnego nie ma on innych minimów lokalnych.


### ZADANIE
(1 pkt.)

Znajdź pochodne funkcji napisanej przez prowadzącego na tablicy korzystając z sympy i kodu powyżej.

Podpowiedź: Druga pochodna $f(x)$ jest pierwszą pochodną jakiej funkcji?

In [ ]:
# Zdefiniuj symbol 'x', aby SymPy wiedziało, że to zmienna
x = sp. symbols('x')

# Zdefiniuj funkcję (f(x))
f_x = sp.cos(x)

# Oblicz pierwszą pochodną (f'(x))
f_derivative = sp.diff(f_x, x)

# Oblicz drugą pochodną (f''(x))
f_second_derivative = sp.diff(f_derivative, x)

print(f"Funkcja f(x): {f_x}")
print(f"Pierwsza pochodna f'(x): {f_derivative}")
print(f"Druga pochodna f''(x): {f_second_derivative}")

## Gradient Descent

Wykorzystanie pochodnej ogranicza nas do funkcji jednej zmiennej. Wiemy jak poruszać się po $x$. Co jednak zrobić, gdy nasza funkcja zależy od czegoś więcej niż samego $x$? Potrzebujemy innego, ale podobnego narzędzia, bardziej uogólnionego. Wprowadźmy zatem nowe pojęcia:

1.  **Pochodna Cząstkowa** ($\frac{\partial f}{\partial x}$): To pochodna funkcji wielu zmiennych względem jednej z nich. Mówi o nachyleniu funkcji gdy poruszamy się tylko wzdłuż jednej osi (np. osi $x$), traktując wszystkie inne (np. $y$, $z$) jak stałe. Gdybyśmy próbowali zejść z góry do doliny, naszymi zmiennymi mogły być np. kierunki północ-południe i wschód-zachód. Chociaż moglibyśmy poruszać się nie tylko w tych dwóch kierunkach, pochodna cząstkowa dla kierunku północ-południe mówiłaby nam, jak zmienia się wysokość góry gdy poruszamy się idealnie w tym kierunku.
2.  **Gradient** ($\nabla f$): To **wektor** złożony ze wszystkich pochodnych cząstkowych. Gradient zawsze wskazuje kierunek **najszybszego wzrostu** funkcji (o ile funkcja jest różniczkowalna w danym punkcie).

$$\nabla f(x, y) = \left[ \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y} \right]$$

Metodę gradientu prostego (Gradient Descent) jest bardzo podobna do metody wykorzystującej pochodną.

1.  Obliczamy gradient w punkcie $\mathbf{p}_i = (x_i, y_i)$ -> $\nabla f(\mathbf{p}_i)$. Wiemy, że wskazuje on kierunek najszybszego wzrostu ("w górę").
2.  Jeśli minimalizujemy, to chcemy iść "w dół", więc robimy krok w **przeciwnym kierunku** niż gradient, tak samo jak robiliśmy krok w kierunku przeciwnym do pochodnej.
3.  Długość tego kroku jest kontrolowana przez **współczynnik uczenia** (ang. *learning rate*, $\eta$) i długość wektora gradientu, tak jak zależała wcześniej od wartości pochodnej.

Wzór jest prosty:

$$
\mathbf{p}_{i+1} = \mathbf{p}_i - \eta \nabla f(\mathbf{p}_i)
$$  

Tak jak dla funkcji jednej zmiennej:
* Jeśli $\eta$ jest za małe, będziemy bardzo powoli (małymi krokami) schodzić z góry.
* Jeśli $\eta$ jest za duże, możemy "przeskoczyć" dolinę i wylądować na górze po jej drugiej drugiej stronie.

### Zadanie
(1 pkt)

Zaimplementuj funkcję aktualizującą pozycję w oparciu o gradient.

In [ ]:
def compute_new_position_with_gradient(pos, gradient_funcs, learning_rate):
    """
    Oblicza nową pozycję punktu, wykonując krok w kierunku przeciwnym do gradientu (Gradient Descent).

    Funkcja oblicza wektor gradientu w obecnym punkcie, korzystając z przekazanej listy
    funkcji pochodnych cząstkowych, a następnie przesuwa punkt o skalowany wektor
    w kierunku spadku wartości funkcji.

    Args:
        pos (np.ndarray): Obecne współrzędne punktu.
        gradient_funcs (list of callable): Lista funkcji obliczających pochodne cząstkowe.
            Każda funkcja musi przyjmować rozpakowane współrzędne `pos` jako argumenty
            (np. f(x, y) dla pos=[x, y]).
        learning_rate (float): Współczynnik uczenia (step size), określający
            jak duży krok zostanie wykonany w jednej iteracji.

    Returns:
        np.ndarray: Nowa pozycja punktu po wykonaniu kroku optymalizacyjnego.
    """
    grad = [f(pos) for f in gradient_funcs]
    grad = np.array(grad)
    return pos - learning_rate * grad


# --- KONFIGURACJA ---
show_trace = True       # Czy rysować linię śladu?
steps = 50              # Liczba klatek
learning_rate = 0.005     # Współczynnik uczenia

# 1. Definicja Matematyczna
x_sym, y_sym = sp.symbols('x y')
# Funkcja: 0.4x^2 + y^2 - cos(2x)
f_sym = 7*x_sym**2 + 7*y_sym**2

# Pochodne (Gradient)
df_dx = sp.diff(f_sym, x_sym)
df_dy = sp.diff(f_sym, y_sym)


# Konwersja na funkcje numeryczne (szybkie obliczenia)
f_num = sp.lambdify([(x_sym, y_sym)], f_sym, 'numpy')
grad_x_num = sp.lambdify([(x_sym, y_sym)], df_dx, 'numpy')
grad_y_num = sp.lambdify([(x_sym, y_sym)], df_dy, 'numpy')

# Scenariusze Startowe
scenarios = [
    {"name": "Lewy",   "start": np.array([-2.0, 1.5]), "color": "red"},
    {"name": "Środkowy",   "start": np.array([-0.5, 1.5]), "color": "white"}, # Biały kolor dla kontrastu na ciemnym tle
    {"name": "Prawy",  "start": np.array([ 2.0, -1.5]), "color": "cyan"}
]

make_2d_parabolic_example(f_num, [grad_x_num, grad_y_num], scenarios, learning_rate, compute_new_position_with_gradient, steps=steps, show_trace=show_trace)


Tak jak w przykładach z pochodną, przypadki gdzie funkcja zbiega do jednego minimum, są rzadkością. Ponownie musimy uważać na minima lokalne.

### ZADANIE
(2 pkt.)

Używając kodu poniżej zbadaj:
* W jaki sposób learning_rate wpływa na działanie algorytmu Gradient Descent dla funkcji wielu zmiennych
* W jaki sposób pozycja początkowa (`start`) wpływa na znajdowane rozwiązania

Zapisz wnioski.

In [ ]:
# --- KONFIGURACJA ---
show_trace = True       # Czy rysować linię śladu?
steps = 50              # Liczba klatek
learning_rate = 0.01   # Współczynnik uczenia

# 1. Definicja Matematyczna
x_sym, y_sym = sp.symbols('x y')
# Funkcja: 0.4x^2 + y^2 - cos(2x)
f_sym = 0.4*x_sym**2 + y_sym**2 - sp.cos(2*x_sym)

# Pochodne (Gradient)
df_dx = sp.diff(f_sym, x_sym)
df_dy = sp.diff(f_sym, y_sym)

# Konwersja na funkcje numeryczne (szybkie obliczenia)
f_num = sp.lambdify([(x_sym, y_sym)], f_sym, 'numpy')
grad_x_num = sp.lambdify([(x_sym, y_sym)], df_dx, 'numpy')
grad_y_num = sp.lambdify([(x_sym, y_sym)], df_dy, 'numpy')

# 2. Scenariusze Startowe
scenarios = [
    {"name": "Lokalne (Lewe)",   "start": np.array([-3.0, 1.5]), "color": "red"},
    {"name": "GLOBALNE",         "start": np.array([-0.8, 1.5]), "color": "white"},
    {"name": "Lokalne (Prawe)",  "start": np.array([ 2.0, -1.5]), "color": "cyan"}
]

make_3D_higher_order_example(f_num, [grad_x_num, grad_y_num], scenarios, learning_rate, compute_new_position_with_gradient, steps=steps, show_trace=show_trace)

Podobnie jak w poprzednim przykładzie learning_rate mocno wpływa na znajdowanie rozwiązań. Przy małym learning rate dla punktów startowych blisko minimów lokalnych, zostają one w minimach lokalnych i nie znajdują minimum globalnego. Natomiast dla dużych learnig_rate algorytm potrafi robić za duże skoki, które cały czas omijają minimum, nawet nie zatrzymują się na lokalnych. Natomiast przy bardzo malym learning rate (np. 0.01) w określonej liczbie kroków algorytm nie zdąża dochodzić do minimum. Znowu podobnie jak poprzednio, pozycja startowa bliżej minimum globalnego, niż innych lokalnych daje zwykle lepszy wynik (znajduje minimum globalne).

### Minimalizacja funkcji kosztu

W pierwszym laboratorium wykorzystywaliśmy funkcję kosztu do oceny jakości dopasowania funkcji do danych. Był to błąd średniokwadratowy (*Mean Squared Error*, MSE) — obliczaliśmy kwadraty różnic między punktami pomiarowymi a wartościami przewidywanymi przez model, a następnie uśrednialiśmy po całym zbiorze danych:

$$MSE = {1 \over n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

Gdzie:

* $n$ – liczba próbek (obserwacji),
* $y_i$ – rzeczywista wartość dla próbki,
* $\hat{y}_i$ – wartość przewidywana przez model dla próbki


Dla każdej próbki do rzeczywistej wartości $y_i$ porównujemy nasze predykcje $\hat{y}_i$. Każda tala próbka w danych jest parą wartości $(x_i, y_i)$. Gdzie jednak gdzie ma zastosowanie $x_i$?

Jeżeli potraktujemy nasze dane jako zbiór punktów wygenerowanych z użyciem funkcji $f$, pary $(x_i, y_i)$ będą opisywały wartości argumentów (wejść) i odpowiadających im wartości funkcji (wyjść) $y_i = f(x_i)$.

Funkcja $f$ jest źródłem naszych danych i zazwyczaj jest nieznana. Często nawet nie mamy pojęcia jak mogłaby wyglądać:
* Jeżeli chcesz zamodelować zmianę temperatura wody w czajniku po zagotowaniu, $x$ może być czasem od jej zagotowania, $y$ wartość temperatury.
* Jeżeli chcesz zamodelować natężenie ruchu na ulicach miasta, $x$ może być godziną, $y$ liczba samochodów.
* Jeżeli chcesz nauczyć model rozróżniać psy i koty ze zdjęć, $x$ będą zdjęcia, $y$ oznaczenia tych zdjęć ("pies" lub "kot").

O ile szkolna fizyka podpowiada nam możliwy kształt funkcji zmian temperatury wody w czajniku, a natężenie ruchu na ulicach wykazuje okresowość (cyklicznie rośnie i maleje), samodzielne wymyślenie jak wartości pikseli na zdjęciu przekładają się na klasę "pies" lub "kot" jest praktycznie niemożliwe.

Z pomocą przychodzi nam optymalizacja. Jeżeli nie znamy kształtu funkcji, możemy spróbować ją znaleźć, albo chociaż przybliżyć, czyli znaleźć takie parametry nowej funkcji $\hat{f}(x_i)$, które sprawią, że dopasowanie do danych będzie jak najlepsze, czyli błąd dopasowania będzie jak najmniejszy. Będziemy zatem minimalizować błąd dopasowania, $MSE$. Przyjmie ono formę:

$$MSE = {1 \over n} \sum_{i=1}^n (y_i - \hat{f}(x_i))^2$$

Jeżeli nasza funkcja $\hat{f}$ będzie np. funkcją kwadratową, czyli $\hat{y_i} = \hat{f}(x_i) = ax^2_i + bx_i + c$, $MSE$ przyjmie postać:

$$MSE = {1 \over n} \sum_{i=1}^n (y_i - (ax^2_i + bx_i + c))^2$$

Co jednak będziemy optymalizować, skoro argumenty $x_i$ są ustalone w zebranych próbkach? We wcześniejszych przykładach szukaliśmy takiej wartości $x$, dla której $y$ był najmniejszy. Teraz będziemy szukać takiej funkcji $\hat{f}$ która minimalizuje błąd. Skoro o dopasowaniu do danych decyduje kształt funkcji, a ten jest definiowany przez jej parametry, musimy optymalizować parametry funkcji. Dla powyższej funkcji kwadratowej, będziemy optymalizować parametry $a, b, c$ - będą one tym samym, czym był $x$ w poprzednich przykładach. Oznacza to, że będziemy poszukiwać takich parametrów $a, b, c$ dla których wartość $MSE$ będzie najmniejsza.

W prawdziwym świecie, a zwłaszcza w SI, nasze poszukiwane funkcje opisujące świat prawie nigdy nie zależą od jednej zmiennej. Zależą od **milionów** zmiennych (tzw. "wag" lub "parametrów" modelu).



### Przykład

Dopasowanie funkcji liniowej: $y=ax+b$ do danych. Zauważ, że algorytm gradientu prostego (Gradient Descent) działa w dziedzinie parametrów funkcji. Wartość błędu zależy od danych pomiarowych i od kształtu funkcji (czyli wartości jej parametrów). Danymi sterować nie możemy, bo to informacja o świecie. Możemy za to sterować parametrami funkcji. Minimalizujemy zatem wartość błędu ($MSE$), dobierając tak parametry $a,b$, aby funkcja jak najlepiej pokrywała się z danymi.

In [ ]:
# Definiujemy symbole dla x, y (dane) oraz współczynników (wagi modelu)
x, y = sp.symbols('x y')
a, b = sp.symbols('a b')
# Nasz model (Hipoteza): Funkcja liniowa
# Model nie zna parametrów i musi je dopiero znaleźć
model_sym = a*x + b

# Funkcja straty: Mean Squared Error (MSE)
loss_sym = (y - model_sym)**2

# Obliczamy Gradienty funkcji straty (MSE)
# SymPy wykonuje różniczkowanie za nas.
gradients_sym = {
    'a': sp.diff(loss_sym, a),
    'b': sp.diff(loss_sym, b)
}

print("Gradient funkcji straty po współczynniku 'a' (przy x):")
print(gradients_sym['a'])

# Konwersja na szybkie funkcje numeryczne
# predict_func: przyjmuje x i współczynniki -> zwraca y
params = (x, a, b)
predict_func = sp.lambdify([params], model_sym, 'numpy')

# grad_funcs: słownik funkcji obliczających gradienty numerycznie
grad_funcs = {}
# params_grad to (x, y, a, b) - potrzebujemy też y do obliczenia błędu
params_grad = (x, y, a, b)
loss_func = sp.lambdify([params_grad], loss_sym, 'numpy')

for key, expr in gradients_sym.items():
    grad_funcs[key] = sp.lambdify([params_grad], expr, 'numpy')

# Inicjalizacja losowych wag (ponieważ gdzieś trzeba zacząć)
weights = {
    'a': -4,
    'b': 6,
}

learning_rate = 0.01 # Mały krok, bo x^6 może produkować ogromne wartości!
iterations = 100

make_linear_3d_example(predict_func, grad_funcs, weights, loss_func, learning_rate, iterations=iterations)

### Przykład

Dopasowanie wielomianu szóstego stopnia do danych.

In [ ]:
# Definiujemy symbole dla x, y (dane) oraz współczynników (wagi modelu)
x, y = sp.symbols('x y')
a, b, c, d, e, f, g = sp.symbols('a b c d e f g')

# Nasz model (Hipoteza): Wielomian 6. stopnia
# Model nie zna parametrów i musi je dopiero znaleźć
model_sym = a*x**6 + b*x**5 + c*x**4 + d*x**3 + e*x**2 + f*x + g

# Funkcja Kosztu dla jednego punktu (Kwadrat różnicy)
loss_sym = (y - model_sym)**2

# Obliczamy Gradienty funkcji straty (MSE)
# SymPy wykonuje różniczkowanie za nas.
gradients_sym = {
    'a': sp.diff(loss_sym, a),
    'b': sp.diff(loss_sym, b),
    'c': sp.diff(loss_sym, c),
    'd': sp.diff(loss_sym, d),
    'e': sp.diff(loss_sym, e),
    'f': sp.diff(loss_sym, f),
    'g': sp.diff(loss_sym, g),
}

print("Gradient po współczynniku 'a' (przy x^6):")
print(gradients_sym['a'])

# Konwersja na szybkie funkcje numeryczne
# predict_func: przyjmuje x i współczynniki -> zwraca y
params = (x, a, b, c, d, e, f, g)
predict_func = sp.lambdify([params], model_sym, 'numpy')

# grad_funcs: słownik funkcji obliczających gradienty numerycznie
grad_funcs = {}
# params_grad to (x, y, a, b, c, d, e, f, g) - potrzebujemy też y do obliczenia błędu
params_grad = (x, y, a, b, c, d, e, f, g)
loss_func = sp.lambdify([params_grad], loss_sym, 'numpy')

for key, expr in gradients_sym.items():
    grad_funcs[key] = sp.lambdify([params_grad], expr, 'numpy')


# Inicjalizacja losowych wag (ponieważ gdzieś trzeba zacząć)
weights = {
    'a': np.random.randn() * 0.1,
    'b': np.random.randn() * 0.1,
    'c': np.random.randn() * 0.1,
    'd': np.random.randn() * 0.1,
    'e': np.random.randn() * 0.1,
    'f': np.random.randn() * 0.1,
    'g': np.random.randn() * 0.1
}

learning_rate = 0.01 # Mały krok, bo x^6 może produkować ogromne wartości!
iterations = 300


make_polynomial_example(predict_func, grad_funcs, weights, loss_func, learning_rate, iterations=iterations)

### ZADANIE
(5 pkt.)

Wykorzystajmy teraz gradient descent w symulacji sytuacji życiowej.

Załóżmy, że szukasz mieszkania we Wrocławiu (w związku ze studiami, pracą, czy z innych powodów). Wykorzystaj poniższy kod do znalezienia optymalnej lokalizacji.

* Zdefiniuj przynajmniej 5 interesujących Cię punktów we Wrocławiu i określ ich wagę (np. ile razy w tygodniu odwiedzasz te punkty)
    * Dla odniesienia w liście dodano "D2 PWr", jednak możesz to zmodyfikować
    * Do szukania koordynatów możesz wykorzystać np. [Google Maps](https://www.google.com/maps)
* Zdefiniuj niezbędne parametry przeszukania
    * Zostały oznaczone przy pomocy `...`
* Wymyśl funkcję kosztu, np. średnią ważoną odległości od punktów
    * Sympy implementuje [wiele funkcji opisanych w dokumentacji](https://docs.sympy.org/latest/modules/functions/elementary.html#sympy.functions.elementary.exponential.log). Możesz z nich skorzystać, np. z `sp.log`.
* Policz pochodne cząstkowe funkcji kosztu, przekonwertuj uzyskane funkcje i spakuj do jednej listy reprezentującej gradient
* Uzupełnij linijkę aktualizującą pozycję. Możesz wykorzystać wcześniej napisaną funkcję `compute_new_position_with_gradient` lub napisać kod od nowa.

Zbadaj wpływ wybranych parametrów przeszukiwania na wyniki i zapisz obserwacje dotyczące działania algorytmu i wnioski.

In [ ]:
def plot_new_home_location(optimal_location, path, attractors, repulsors, radius_meters=1000):
    # 4. WIZUALIZACJA (FOLIUM)
    m = folium.Map(location=optimal_location, zoom_start=13)

    # A. Rysujemy punkty zdefiniowane (Cele)
    for p in attractors:
        folium.Marker(
            location=p["coords"],
            popup=f"{p['name']} (Waga: {p['weight']})",
            icon=folium.Icon(color="blue", icon="info-sign")
        ).add_to(m)
        # Linia od celu do optimum (wizualizacja "napięcia")
        folium.PolyLine(
            locations=[p["coords"], optimal_location],
            color="gray", weight=1, opacity=0.5, dash_array='5, 5'
        ).add_to(m)

    for p in repulsors:
        folium.Marker(
            location=p["coords"],
            popup=f"{p['name']} (Waga: {p['weight']})",
            icon=folium.Icon(color="red", icon="info-sign")
        ).add_to(m)
        # Linia od celu do optimum (wizualizacja "napięcia")
        folium.PolyLine(
            locations=[p["coords"], optimal_location],
            color="gray", weight=1, opacity=0.5, dash_array='5, 5'
        ).add_to(m)


    # B. Rysujemy ścieżkę Gradient Descent
    folium.PolyLine(
        locations=path,
        color="red", weight=3, opacity=0.7,
        tooltip="Ścieżka optymalizacji"
    ).add_to(m)

    folium.Marker(
        location=optimal_location,
        popup="<b>Optymalne miejsce</b>",
        icon=folium.Icon(color="green", icon="home", prefix='fa')
    ).add_to(m)

    folium.Circle(
        location=optimal_location,
        radius=radius_meters,
        color="green",
        fill=True,
        fill_opacity=0.2,
        popup=f"Obszar akceptowalny ({radius_meters}m)"
    ).add_to(m)

    return m

In [ ]:
map_file_name = "wroclaw_optimal_home.html"

# KONFIGURACJA DANYCH
# Format: [Szerokość (lat), Długość (lon), Waga (ile razy w tygodniu)]
attractors = [
        {"name": "Politechnika Wrocławska bud. D2", "coords": [51.1099, 17.0569], "weight": 5},
        {"name": "C. H. Korona", "coords": [51.141483084088826, 17.085826529438954], "weight": 2},
        {"name": "Karłowice", "coords": [51.136864040810245, 17.042250917212385], "weight": 2},
        {"name": "Ostrów Tumski", "coords": [51.11428274514853, 17.044672439706833], "weight": 5},
        {"name": "Park Szczytnicki", "coords": [51.11481121803854, 17.082538211087794], "weight": 3},
    ]

repulsors = [
        {"name": "Rynek (Hałas i koszt)", "coords": [51.1100, 17.0321], "weight": 0.00002},
        {"name": "Pasaż Niepolda (Hałas)", "coords": [51.110088832149586, 17.0254785797491], "weight": 0.00002},
        {"name": "Psie Pole", "coords": [51.1451899473138, 17.11685969599803], "weight": 0.00005},
        {"name": "Jagodno", "coords": [51.055986197953445, 17.058021801883818], "weight": 0.00005},
    ]


# Zdefiniuj parametry przeszukiwania
learning_rate = 0.0025  # Mały krok, bo koordynaty to małe liczby
iterations = 100 # Liczba iteracji wynanych przez algorytm
radius_meters = 999 # Promień akceptowalnych lokalizacji (np. 1km od ideału)

# Zdefiniuj funkcję kosztu w oparciu o koordynaty i wagę lokalizacji
# arr = sum([abs(a["coords"][0]-x)*a["weight"] for a in attractors]) + sum([abs(a["coords"][1]-y)*a["weight"] for a in attractors])

# Zdefiniuj zmienne symboliczne: x (szerokość), y (długość)
x, y = sp.symbols('x y')

# Zdefiniuj funkcję kosztu
# Jeżeli chcesz później wykorzystać napisaną przez siebie funkcję `compute_new_position_with_gradient`,
# mozesz uzupełnić przykład ponizej
cost_function = 0
# Obliczamy ważony koszt dla każdego punktu przyciągającego
for p in attractors:
    p_lat, p_lon = p["coords"]
    w = p["weight"]
    cost_function += w * (sp.sqrt(((p_lat - x)**2) + ((p_lon - y)**2)))
epsilon = 1e-9
for r in repulsors:
    r_lat, r_lon = r["coords"]
    s = r["weight"]
    # Aby unikać - im bliżej, tym koszt musi rosnąć szybciej
    cost_function += s / sp.sqrt(((r_lat - x)**2) + ((r_lon - y)**2))

# Policz pochodne i zapisz do zmiennych
grad_x = sp.diff(cost_function, x)
print(f"Wyprowadzony wzór na gradient X (szerokości geograficznej): {grad_x}")
grad_y = sp.diff(cost_function, y)
print(f"Wyprowadzony wzór na gradient Y (długości geograficznej): {grad_y}")

# Przekonwertuj funkcje pochodnych
calc_grad_x = sp.lambdify([(x, y)], grad_x, 'numpy')
calc_grad_y = sp.lambdify([(x, y)], grad_y, 'numpy')
# Zapisz pochodne do jednej listy - gradientu
grad = [calc_grad_x, calc_grad_y]

# Punkt startowy (gdzieś na uboczu, żeby zobaczyć jak algorytm wędruje)
current_pos = np.array([51.1250, 17.0200])

path = [current_pos.copy()]

# Poszukiwanie lokalizacji z użyciem gradient descent
for i in range(iterations):
    # Obliczamy gradient w obecnym punkcie
    # Aktualizacja pozycji: wykorzystaj napisaną wcześniej
    # funkcję compute_new_position_with_gradient
    current_pos = compute_new_position_with_gradient(current_pos, grad, learning_rate)
    path.append(current_pos.copy())

optimal_location = current_pos
print(f"Znaleziono optymalną lokalizację: {optimal_location}")


# Uruchomienie i zapisanie mapy
map_result = plot_new_home_location(optimal_location, path, attractors, repulsors, radius_meters)
print(f"Zapisuję mapę: {map_file_name}")
map_result.save(map_file_name)

Źródła:

1. Fiske, Susan T. Tufts, and Shelley E. Taylor. "Social cognition: From brains to culture." (2020): 1-672.
2. Kahneman, Daniel. Thinking, fast and slow. macmillan, 2011.